# TRISTAR GTFS Batch Ingestion

Loads static GTFS snapshot into Bronze Delta tables.

## 1. Configuration

Define Unity Catalog and Volume paths for batch ingestion.

In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("schema_bronze", "team_tristar_bronze")
dbutils.widgets.text("volume", "raw_data")

catalog = dbutils.widgets.get("catalog")
schema_bronze = dbutils.widgets.get("schema_bronze")
volume = dbutils.widgets.get("volume")

In [0]:
base_path = f"/Volumes/{catalog}/{schema_bronze}/{volume}"

batch_path = f"{base_path}/batch"
archive_path = f"{batch_path}/archive"
gtfs_archive_path = f"{archive_path}/gtfs"
extracted_path = f"{batch_path}/extracted"


## 2. Source Configuration

Configure the TRISTAR GTFS source and CKAN metadata endpoint.

In [0]:
import requests
import zipfile
from datetime import datetime

resource_id = "30e783e4-2bec-4a7d-bb22-ee3e3b26ca96"

gtfs_url = (
    "https://ckan.multimediagdansk.pl/dataset/"
    "c24aa637-3619-4dc2-a171-a23eec8f2172/resource/"
    f"{resource_id}/download/gtfsgoogle.zip"
)

metadata_url = (
    "https://ckan.multimediagdansk.pl/api/3/action/"
    f"resource_show?id={resource_id}"
)

In [0]:
dbutils.fs.mkdirs(archive_path)
dbutils.fs.mkdirs(extracted_path)


## 3. Source Update Check

Check the source update date and download a new snapshot only when needed.

In [0]:
metadata_response = requests.get(metadata_url, timeout=30)
metadata_response.raise_for_status()

metadata = metadata_response.json()["result"]

source_update_timestamp = metadata["last_modified"]

source_update_date = datetime.fromisoformat(
    source_update_timestamp
).strftime("%Y-%m-%d")

In [0]:
zip_name = f"gtfs_{source_update_date}.zip"

zip_path = f"{gtfs_archive_path}/{zip_name}"
extract_path = f"{extracted_path}/{source_update_date}"

In [0]:
existing_archives = {
    file.name
    for file in dbutils.fs.ls(gtfs_archive_path)
}

new_snapshot = zip_name not in existing_archives

In [0]:
if new_snapshot:
    response = requests.get(gtfs_url, timeout=120)
    response.raise_for_status()

    with open(zip_path, "wb") as file:
        file.write(response.content)

## 4. Extract and Validate Files

Extract the GTFS snapshot and validate the required source files.

In [0]:
expected_files = {
    "calendar_dates.txt",
    "routes.txt",
    "stop_times.txt",
    "stops.txt",
    "trips.txt"
}

dbutils.fs.mkdirs(extract_path)

actual_files = {
    file.name
    for file in dbutils.fs.ls(extract_path)
}

if not expected_files.issubset(actual_files):
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

actual_files = {
    file.name
    for file in dbutils.fs.ls(extract_path)
}

missing_files = expected_files - actual_files

if missing_files:
    raise ValueError(
        f"Missing GTFS files: {sorted(missing_files)}"
    )

## 5. Bronze Ingestion

Load selected GTFS files into Bronze Delta tables using a full overwrite.

In [0]:
from pyspark.sql.functions import lit, current_timestamp

tables = [
    "calendar_dates",
    "routes",
    "stop_times",
    "stops",
    "trips"
]

for table_name in tables:

    table = f"{catalog}.{schema_bronze}.{table_name}"
    file_path = f"{extract_path}/{table_name}.txt"

    if spark.catalog.tableExists(table):
        loaded_date = spark.table(table).first()["source_update_date"]

        if str(loaded_date) == source_update_date:
            continue

    df = spark.read.option("header", "true").csv(file_path)

    bronze_df = (
        df
        .withColumn(
            "source",
            lit(f"tristar_gtfs/{table_name}.txt")
        )
        .withColumn(
            "source_update_date",
            lit(source_update_date).cast("date")
        )
        .withColumn(
            "ingestion_timestamp",
            current_timestamp()
        )
    )

    (
        bronze_df.write
            .format("delta")
            .mode("append")
            .option("overwriteSchema", "true")
            .saveAsTable(table)
    )